# 02 · Embedding behavior and limits

## The engineering question
Does a high semantic similarity score establish that two statements have the same meaning or answer the same question?

**Learning outcomes:** inspect tokens and truncation; recognize negation and identifier failures; understand normalization and query/document roles; distinguish model configuration from application correctness.

The default MiniLM encoder maps text to 384 dimensions and normalizes output. It has a configured 256-token limit. Unlike some asymmetric retrieval models, this model uses no special query/document prefix. The API still makes the role explicit so changing model families cannot silently omit required prefixes.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("02", "B", docker=False)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from vector_masterclass.models import embed, dense_model
from vector_masterclass.config import model_config

cfg = model_config("dense")
texts = [
    "Atlas supports power over Ethernet.",
    "Atlas does not support power over Ethernet.",
    "Atlas error E104 means a disconnected cable.",
    "Atlas error E140 means a configuration signature failure.",
    "How long is the warranty?",
    "The warranty lasts two years.",
]
vectors = embed(texts)
similarities = vectors @ vectors.T
frame = lesson.table(
    pd.DataFrame(similarities, columns=[f"S{i}" for i in range(len(texts))]), "similarity"
)
display(pd.DataFrame({"id": [f"S{i}" for i in range(len(texts))], "text": texts}))
fig, ax = plt.subplots(figsize=(7, 5))
picture = ax.imshow(similarities, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(texts)), [f"S{i}" for i in range(len(texts))])
ax.set_yticks(range(len(texts)), [f"S{i}" for i in range(len(texts))])
ax.set_title("Topical similarity can survive a change in factual meaning")
fig.colorbar(picture, ax=ax, label="Cosine similarity")
lesson.figure(fig)
plt.show()

In [ ]:
tokenizer = dense_model().tokenizer
filler = "Routine maintenance requires careful inspection. " * 100
late_fact = filler + " The warranty lasts two years."
early_fact = "The warranty lasts two years. " + filler
truncated_late = tokenizer(late_fact, truncation=True, max_length=cfg["max_length"])["input_ids"]
decoded = tokenizer.decode(truncated_late)
token_table = lesson.table(
    pd.DataFrame(
        [
            {
                "placement": name,
                "untruncated_tokens": len(tokenizer.encode(text)),
                "encoded_tokens": len(
                    tokenizer(text, truncation=True, max_length=cfg["max_length"])["input_ids"]
                ),
            }
            for name, text in [("late", late_fact), ("early", early_fact)]
        ]
    ),
    "truncation",
)
display(token_table)
print("Warranty present after late-fact truncation:", "warranty" in decoded.lower())

## Interpretation and exercise
Inspect the negation pair rather than assuming a score threshold makes it safe to merge or cache answers. A high score reflects the encoder's learned geometry, not a calibrated probability that a claim is true. The truncation example is structural: content beyond the configured token limit never reaches the encoder.

**Exercise:** move the warranty sentence earlier in the text and compare query similarity. Then replace `E104` with `E140` while keeping the surrounding words identical. Which failures require lexical matching, metadata, chunking, or a reranker?

In production, pin tokenizer/model revisions and normalization together. Do not combine embeddings from different models merely because both have 384 dimensions. Choose a task-specific encoder and validate on your workload; use exact identifiers when the task demands them.

In [ ]:
manifest = lesson.finish(
    {
        "dimension_contract": vectors.shape == (6, 384),
        "unit_norms": bool(np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5)),
        "late_fact_truncated": "warranty" not in decoded.lower(),
        "explicit_role_configuration": cfg["query_prefix"] == "" and cfg["document_prefix"] == "",
    },
    observations={"negation_cosine": float(similarities[0, 1]), "model": cfg},
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.